# 02 Terminal Ladder

This notebook is the second evidence notebook in the public release suite.

## Purpose

It reproduces the **terminal-band comparator** on the final **28.5B to 31.5B** segment and shows that:

- the dominant organizing coordinate remains **`h1`**
- the unusually tight terminal locking persists at **52k, 8k, and 1k**
- the **8k** board yields the strongest concentration of the three tested terminal scales

This notebook is **artifact-first**. By default it loads frozen terminal summary JSON files and rebuilds the headline comparator table and plot without rerunning the heavy extraction pipeline.

## Robustness note

Some users may not have a standalone terminal **8k trimmed summary JSON** saved under the original `r7_...` filename.  
This release notebook therefore supports a fallback path:

- preferred: terminal 8k trimmed summary JSON
- fallback: terminal 8k S6 matched-null summary JSON for the 8k harmonic/concentration row

That keeps the public ladder notebook usable even when the internal 8k trimmed-summary filename was not frozen separately.


## Reading note

This notebook supports the terminal-band claims in the paper.

For the rest of the public evidence chain, continue with:

- `03_terminal_8k_matched_packet_null.ipynb`
- `04_terminal_8k_jackknife.ipynb`


In [ ]:
# Optional path settings for Colab or local runs

import os
from pathlib import Path

DEFAULT_OUT_DIR = "/content/drive/MyDrive/Colab Notebooks/ForgeV16c"
OUT_DIR = os.environ.get("FORGE_V16C_OUT_DIR", DEFAULT_OUT_DIR)

print("OUT_DIR =", OUT_DIR)

In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def first_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

FILES = {
    "52k": {
        "q1_summary": first_existing([
            os.path.join(OUT_DIR, "q1_terminal_continuity_summary_v16c_52000_31p5B.json"),
        ]),
        "trimmed_summary": first_existing([
            os.path.join(OUT_DIR, "r6_terminal_trimmed_summary_v16c_52000_31p5B.json"),
            os.path.join(OUT_DIR, "R6_terminal_trimmed_summary_v16c_52000_31p5B.json"),
        ]),
        "fallback_s6_summary": None,
        "forward_csv": first_existing([
            os.path.join(OUT_DIR, "r6_forward_predictions_v16c_52000_31p5B.csv"),
            os.path.join(OUT_DIR, "R6_forward_predictions_v16c_52000_31p5B.csv"),
        ]),
    },
    "8k": {
        "q1_summary": first_existing([
            os.path.join(OUT_DIR, "q1_terminal_continuity_summary_v16c_8000_31p5B.json"),
        ]),
        "trimmed_summary": first_existing([
            os.path.join(OUT_DIR, "r7_terminal_trimmed_summary_v16c_8000_31p5B.json"),
            os.path.join(OUT_DIR, "R7_terminal_trimmed_summary_v16c_8000_31p5B.json"),
            *sorted(glob.glob(os.path.join(OUT_DIR, "*terminal*trimmed*8000*31p5B*.json"))),
            *sorted(glob.glob(os.path.join(OUT_DIR, "*8000*trimmed*summary*.json"))),
        ]),
        "fallback_s6_summary": first_existing([
            os.path.join(OUT_DIR, "s6_terminal8k_null_overnight_v1_summary.json"),
        ]),
        "forward_csv": None,
    },
    "1k": {
        "q1_summary": first_existing([
            os.path.join(OUT_DIR, "q1_terminal_continuity_summary_v16c_1000_31p5B.json"),
        ]),
        "trimmed_summary": first_existing([
            os.path.join(OUT_DIR, "r5_terminal_trimmed_summary_v16c_1000_31p5B.json"),
            os.path.join(OUT_DIR, "R5_terminal_trimmed_summary_v16c_1000_31p5B.json"),
        ]),
        "fallback_s6_summary": None,
        "forward_csv": first_existing([
            os.path.join(OUT_DIR, "r5_forward_predictions_v16c_1000_31p5B.csv"),
            os.path.join(OUT_DIR, "R5_forward_predictions_v16c_1000_31p5B.csv"),
        ]),
    },
}

for label, paths in FILES.items():
    print(f"--- {label} ---")
    for k, p in paths.items():
        status = "SKIPPED" if p is None and k == "forward_csv" else ("FOUND" if p else "MISSING")
        print(f"{k:>18}: {status}")
        if p:
            print(f"                    {p}")

## Load the terminal summaries

This cell builds a compact terminal-ladder table from the frozen artifacts.

- 52k and 1k are expected to load from Q1 + trimmed-summary pairs
- 8k will use the trimmed-summary if present
- if the 8k trimmed-summary is missing, the notebook falls back to the frozen S6 summary for the 8k harmonic/concentration row


In [ ]:
rows = []

for label, paths in FILES.items():
    if not paths["q1_summary"]:
        raise FileNotFoundError(f"[{label}] Missing q1_summary")

    q1 = load_json(paths["q1_summary"])

    forward_b1 = np.nan
    forward_b2 = np.nan
    if paths["forward_csv"] and os.path.exists(paths["forward_csv"]):
        fdf = pd.read_csv(paths["forward_csv"])
        if len(fdf) > 0:
            forward_b1 = fdf.iloc[0].get("best_b1_pred", np.nan)
            forward_b2 = fdf.iloc[0].get("best_b2_pred", np.nan)

    row = {
        "chunk_label": label,
        "rows_built": q1.get("rows_built"),
        "altitude_min_m": q1.get("altitude_min_m"),
        "altitude_max_m": q1.get("altitude_max_m"),
        "q1_rmse_pure": q1.get("rmse_pure"),
        "q1_rmse_fit": q1.get("rmse_fit"),
        "q1_boundary_cross_rows": q1.get("boundary_cross_rows"),
        "q1_near_edge_rows": q1.get("near_edge_rows"),
        "q1_pure_interior_rows": q1.get("pure_interior_rows"),
        "q1_fitted_a": q1.get("fitted_a"),
        "q1_fitted_b": q1.get("fitted_b"),
        "forward_b1_step1": forward_b1,
        "forward_b2_step1": forward_b2,
        "data_source": None,
    }

    if paths["trimmed_summary"]:
        tr = load_json(paths["trimmed_summary"])
        row.update({
            "packet_count": tr.get("packet_count"),
            "eligible_row_count": tr.get("eligible_row_count"),
            "compression_threshold": tr.get("compression_threshold"),
            "best_candidate": tr.get("best_candidate"),
            "best_harmonic": tr.get("best_harmonic"),
            "best_resultant_r": tr.get("best_resultant_r"),
            "best_arc80": tr.get("best_arc80"),
            "rolling_windows": tr.get("rolling_windows"),
            "best_b1_model": tr.get("best_b1_model"),
            "best_b2_model": tr.get("best_b2_model"),
            "median_center_step_m": tr.get("median_center_step_m"),
            "data_source": "trimmed_summary",
        })
    elif label == "8k" and paths["fallback_s6_summary"]:
        s6 = load_json(paths["fallback_s6_summary"])
        row.update({
            "packet_count": s6.get("real_packet_count"),
            "eligible_row_count": np.nan,
            "compression_threshold": s6.get("threshold"),
            "best_candidate": s6.get("real_best_candidate"),
            "best_harmonic": s6.get("real_best_harmonic"),
            "best_resultant_r": s6.get("real_best_resultant_r"),
            "best_arc80": s6.get("real_best_arc80"),
            "rolling_windows": np.nan,
            "best_b1_model": np.nan,
            "best_b2_model": np.nan,
            "median_center_step_m": np.nan,
            "data_source": "s6_fallback_summary",
        })
    else:
        raise FileNotFoundError(f"[{label}] Missing trimmed_summary and no supported fallback artifact found")

    rows.append(row)

terminal_df = pd.DataFrame(rows)
order = ["52k", "8k", "1k"]
terminal_df["chunk_label"] = pd.Categorical(terminal_df["chunk_label"], order, ordered=True)
terminal_df = terminal_df.sort_values("chunk_label").reset_index(drop=True)

display(terminal_df)

## Headline interpretation

The terminal ladder supports three release-level claims:

1. **Terminal-band persistence**  
   The same dominant harmonic, `h1`, survives across 52k, 8k, and 1k on the same band.

2. **Local tightening**  
   The terminal field is much tighter than the broad-board regime.

3. **Intermediate-scale maximum**  
   The 8k board yields the strongest concentration of the three tested terminal scales, rather than the effect being trivially strongest only on the coarsest or finest board.


In [ ]:
headline_cols = [
    "chunk_label",
    "rows_built",
    "best_candidate",
    "best_resultant_r",
    "best_arc80",
    "best_b1_model",
    "best_b2_model",
    "forward_b1_step1",
    "forward_b2_step1",
    "data_source",
]
display(terminal_df[headline_cols])

## Plot the terminal ladder

This plot is a compact visual summary of the 52k / 8k / 1k comparator.


In [ ]:
plot_df = terminal_df.copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

ax = axes[0]
ax.plot(plot_df["chunk_label"].astype(str), plot_df["best_resultant_r"], marker="o")
ax.set_title("A. Terminal best resultant R")
ax.set_xlabel("Terminal row size")
ax.set_ylabel("Resultant R")
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(plot_df["chunk_label"].astype(str), plot_df["best_arc80"], marker="o")
ax.set_title("B. Terminal best Arc80")
ax.set_xlabel("Terminal row size")
ax.set_ylabel("Arc80")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Simple release checks

These are the release-level checks this notebook should satisfy:

- `best_candidate` is `logT_over_pi_h1` at all three terminal scales
- the 8k board has the highest `best_resultant_r`
- the 8k board has the smallest `best_arc80`

If the standalone 8k trimmed-summary is available, the notebook also checks the 8k quadratic model winners directly.  
If the notebook is using the S6 fallback for the 8k row, it skips the 8k model-family check and reports that clearly.


In [ ]:
checks = pd.DataFrame({
    "chunk_label": terminal_df["chunk_label"].astype(str),
    "h1_winner": terminal_df["best_candidate"].eq("logT_over_pi_h1"),
})

display(checks)

all_h1 = checks["h1_winner"].all()
best_r_at_8k = terminal_df.loc[terminal_df["best_resultant_r"].idxmax(), "chunk_label"] == "8k"
best_arc_at_8k = terminal_df.loc[terminal_df["best_arc80"].idxmin(), "chunk_label"] == "8k"

print("All harmonic checks:", all_h1)
print("Highest terminal R at 8k :", best_r_at_8k)
print("Smallest terminal Arc80 at 8k:", best_arc_at_8k)

if terminal_df.loc[terminal_df["chunk_label"] == "8k", "data_source"].iloc[0] == "trimmed_summary":
    model_checks = pd.DataFrame({
        "chunk_label": terminal_df["chunk_label"].astype(str),
        "b1_quadratic": terminal_df["best_b1_model"].eq("poly_deg2"),
        "b2_quadratic": terminal_df["best_b2_model"].eq("poly_deg2"),
    })
    display(model_checks)
    all_models = model_checks[["b1_quadratic", "b2_quadratic"]].all().all()
    print("All quadratic model checks:", all_models)
else:
    print("8k row is using the S6 fallback summary. 8k model-family checks skipped in this public ladder notebook.")
    all_models = terminal_df.loc[terminal_df["chunk_label"].isin(["52k", "1k"]), ["best_b1_model", "best_b2_model"]].eq("poly_deg2").all().all()
    print("52k/1k quadratic model checks:", all_models)

if all_h1 and best_r_at_8k and best_arc_at_8k and all_models:
    print("All terminal-ladder release checks passed.")
else:
    print("One or more terminal-ladder checks failed. Inspect the frozen summary files.")

## Optional export

If you want to save the compact terminal comparator table into the repository artifacts folder, run the next cell.


In [ ]:
# Optional export
# export_path = os.path.join(OUT_DIR, "release_terminal_ladder_summary.csv")
# terminal_df.to_csv(export_path, index=False)
# print("Saved:", export_path)

## Next notebook

Continue to:

**`03_terminal_8k_matched_packet_null.ipynb`**

That notebook reproduces the 2048-replicate matched-packet null and shows that the terminal 8k concentration is not explained by packet morphology alone.
